In [1]:
# Project Variables
lakehouse_name = "lh_dev_ecommerce"
env_name = "dev"
job_id = 101

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 3, Finished, Available, Finished, False)

In [2]:
# Create Project Schemas

spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_metadata_db""")

spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_bronze_db""")

spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_audit_db""")

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
# Create Landing Metadata Table

spark.sql(f"""
CREATE OR REPLACE TABLE {lakehouse_name}.{env_name}_metadata_db.landing_metadata_tbl
(
    JobID INT,
    LandingFilePath STRING,
    LandingFileName STRING,
    BronzeTableName STRING
)
""")

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 5, Finished, Available, Finished, False)

DataFrame[]

In [4]:
# Insert Metadata

spark.sql(f"""
INSERT INTO {lakehouse_name}.{env_name}_metadata_db.landing_metadata_tbl
VALUES
(101,'Files/landing/','customers.csv','customers'),
(101,'Files/landing/','orders.csv','orders'),
(101,'Files/landing/','products.csv','products'),
(101,'Files/landing/','order_items.csv','order_items')
""")

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
# Create Audit Log Table

spark.sql(f"""
CREATE OR REPLACE TABLE {lakehouse_name}.{env_name}_audit_db.job_event_log_tbl
(
    JobID INT,
    JobName STRING,
    TableName STRING,
    Status STRING,
    RowsProcessed INT,
    StartTime TIMESTAMP,
    EndTime TIMESTAMP,
    Message STRING
)
""")

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 7, Finished, Available, Finished, False)

DataFrame[]

In [6]:
# Read Metadata Table

metadata_df = spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_metadata_db.landing_metadata_tbl
WHERE JobID = {job_id}
""")

# metadata_df.show()

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 8, Finished, Available, Finished, False)

In [7]:
metadata_list = metadata_df.collect()

metadata_list

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 9, Finished, Available, Finished, False)

[Row(JobID=101, LandingFilePath='Files/landing/', LandingFileName='order_items.csv', BronzeTableName='order_items'),
 Row(JobID=101, LandingFilePath='Files/landing/', LandingFileName='customers.csv', BronzeTableName='customers'),
 Row(JobID=101, LandingFilePath='Files/landing/', LandingFileName='products.csv', BronzeTableName='products'),
 Row(JobID=101, LandingFilePath='Files/landing/', LandingFileName='orders.csv', BronzeTableName='orders')]

In [8]:
# Read each metadata record

for row in metadata_list:
    print("Landing Path :", row["LandingFilePath"])
    print("Landing File :", row["LandingFileName"])
    print("Bronze Table :", row["BronzeTableName"])
    print("-"*50)

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 10, Finished, Available, Finished, False)

Landing Path : Files/landing/
Landing File : order_items.csv
Bronze Table : order_items
--------------------------------------------------
Landing Path : Files/landing/
Landing File : customers.csv
Bronze Table : customers
--------------------------------------------------
Landing Path : Files/landing/
Landing File : products.csv
Bronze Table : products
--------------------------------------------------
Landing Path : Files/landing/
Landing File : orders.csv
Bronze Table : orders
--------------------------------------------------


In [9]:
from pyspark.sql.functions import current_timestamp

for row in metadata_list:

    file_path = row["LandingFilePath"] + row["LandingFileName"]
    bronze_table = row["BronzeTableName"]

    print(f"Reading : {file_path}")

    df = (
        spark.read
        .option("header", "true")
        .csv(file_path)
    )

    df = df.withColumn("InsertTimestamp", current_timestamp())

    # print(f"Rows : {df.count()}")

    # df.show(5)

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 11, Finished, Available, Finished, False)

Reading : Files/landing/order_items.csv
Rows : 38
Reading : Files/landing/customers.csv
Rows : 29
Reading : Files/landing/products.csv
Rows : 25
Reading : Files/landing/orders.csv
Rows : 27


In [10]:
from pyspark.sql.functions import current_timestamp

for row in metadata_list:

    file_path = row["LandingFilePath"] + row["LandingFileName"]
    bronze_table = row["BronzeTableName"]

    print(f"Loading {file_path} into {bronze_table}")

    df = (
        spark.read
        .option("header", "true")
        .csv(file_path)
    )

    df = df.withColumn("InsertTimestamp", current_timestamp())

    df.write \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"{lakehouse_name}.{env_name}_bronze_db.{bronze_table}")

    print(f"{bronze_table} loaded successfully")

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 12, Finished, Available, Finished, False)

Loading Files/landing/order_items.csv into order_items
order_items loaded successfully
Loading Files/landing/customers.csv into customers
customers loaded successfully
Loading Files/landing/products.csv into products
products loaded successfully
Loading Files/landing/orders.csv into orders
orders loaded successfully


In [11]:
for row in metadata_list:

    bronze_table = row["BronzeTableName"]

    cnt = spark.sql(f"""
        SELECT COUNT(*) AS TotalRows
        FROM {lakehouse_name}.{env_name}_bronze_db.{bronze_table}
    """)

    # print(bronze_table)
    # cnt.show()

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 13, Finished, Available, Finished, False)

In [12]:
from pyspark.sql import Row
from datetime import datetime
from pyspark.sql.types import *

audit_data = []

for row in metadata_list:

    bronze_table = row["BronzeTableName"]

    row_count = spark.table(
        f"{lakehouse_name}.{env_name}_bronze_db.{bronze_table}"
    ).count()

    audit_data.append(
        (
            int(job_id),
            "Landing_to_Bronze",
            bronze_table,
            "Success",
            int(row_count),
            datetime.now(),
            datetime.now(),
            f"{bronze_table} loaded successfully"
        )
    )

schema = StructType([
    StructField("JobID", IntegerType(), True),
    StructField("JobName", StringType(), True),
    StructField("TableName", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("RowsProcessed", IntegerType(), True),
    StructField("StartTime", TimestampType(), True),
    StructField("EndTime", TimestampType(), True),
    StructField("Message", StringType(), True)
])

audit_df = spark.createDataFrame(audit_data, schema=schema)

audit_df.write \
.mode("append") \
.saveAsTable(f"{lakehouse_name}.{env_name}_audit_db.job_event_log_tbl")

StatementMeta(, 0615c78a-60b8-4c12-adb0-a4fff7a61361, 14, Finished, Available, Finished, False)